1.Import thư viện

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

2. Base path

In [9]:
BASE = Path(r"C:\Users\Admin\Desktop\Topic_13_Retail_Store_Sales_Time_Series")

TRAIN_FINAL_PATH = BASE / "src" / "data" / "train_final.csv"
VAL_FEATURES_PATH = BASE / "src" / "data" / "processed" / "splits" / "val_features.csv"
VAL_TARGET_PATH = BASE / "src" / "data" / "processed" / "splits" / "val_target_original.csv"

OUT_DIR = BASE / "src" / "data" / "processed" / "splits"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VAL_RECONSTRUCTED_PATH = OUT_DIR / "val_reconstructed_with_family.csv"
REPRESENTATIVE_PATH = OUT_DIR / "representative_5_pairs.csv"
VAL_5_PAIRS_PATH = OUT_DIR / "val_5_pairs_reconstructed.csv"

3. Đọc dữ liệu

In [10]:
train_final = pd.read_csv(TRAIN_FINAL_PATH, parse_dates=["date"])
val_features = pd.read_csv(VAL_FEATURES_PATH)
val_target = pd.read_csv(VAL_TARGET_PATH)

print("train_final shape :", train_final.shape)
print("val_features shape:", val_features.shape)
print("val_target shape  :", val_target.shape)

train_final shape : (3000888, 51)
val_features shape: (26730, 43)
val_target shape  : (26730, 1)


4. Kiểm tra cột bắt buộc

In [11]:
required_train = ["store_nbr", "family", "date", "year", "month", "day", "store_family_te"]
for c in required_train:
    if c not in train_final.columns:
        raise ValueError(f"train_final.csv thiếu cột bắt buộc: {c}")

required_val_feat = ["store_nbr", "year", "month", "day", "store_family_te"]
for c in required_val_feat:
    if c not in val_features.columns:
        raise ValueError(f"val_features.csv thiếu cột bắt buộc: {c}")

5. Lấy metadata validation từ cuối train_final

In [15]:
val_meta = train_final.tail(n_val).copy().reset_index(drop=True)

print("val_meta shape:", val_meta.shape)
display(val_meta[["store_nbr", "family", "date"]].head())
display(val_meta[["store_nbr", "family", "date"]].tail())

val_meta shape: (26730, 51)


,store_nbr,family,date
0,1,AUTOMOTIVE,2017-08-01
1,1,BABY CARE,2017-08-01
2,1,BEAUTY,2017-08-01
3,1,BEVERAGES,2017-08-01
4,1,BOOKS,2017-08-01


,store_nbr,family,date
26725,9,POULTRY,2017-08-15
26726,9,PREPARED FOODS,2017-08-15
26727,9,PRODUCE,2017-08-15
26728,9,SCHOOL AND OFFICE SUPPLIES,2017-08-15
26729,9,SEAFOOD,2017-08-15


6. Kiểm tra xem tail(n_val) có khớp với val_features không

In [ ]:

store_match = (val_features["store_nbr"].values == val_meta["store_nbr"].values).mean()
print("store_nbr match:", store_match)

if "year" in val_features.columns and "year" in val_meta.columns:
    print("year match    :", (val_features["year"].values == val_meta["year"].values).mean())

if "month" in val_features.columns and "month" in val_meta.columns:
    print("month match   :", (val_features["month"].values == val_meta["month"].values).mean())

if "day" in val_features.columns and "day" in val_meta.columns:
    print("day match     :", (val_features["day"].values == val_meta["day"].values).mean())

=== CHECK MATCH RATE ===
store_nbr match: 1.0
year match    : 0.0
month match   : 0.0
day match     : 0.0


7. Khôi phục validation hoàn chỉnh theo index

In [17]:
val_reconstructed = val_features.copy().reset_index(drop=True)

# gắn metadata từ train_final
val_reconstructed["store_nbr_from_train"] = val_meta["store_nbr"].values
val_reconstructed["family"] = val_meta["family"].values
val_reconstructed["date"] = val_meta["date"].values

print("val_reconstructed shape:", val_reconstructed.shape)
display(val_reconstructed.head())

val_reconstructed shape: (26730, 46)


,store_nbr,onpromotion,oil_price,oil_price_lag_7,oil_price_lag_14,oil_price_rolling_mean_28,oil_price_change_pct,cluster,store_type_encoded,city_freq,...,lag_14,lag_28,lag_364,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7,store_nbr_from_train,family,date
0,1,0,49.31,49.64,53.19,51.235,-0.006648,13,3,0.333333,...,41.0,27.0,6.0,29.285714,31.071429,39.857143,17.114461,1,AUTOMOTIVE,2017-08-01
1,1,0,49.31,49.64,53.19,51.235,-0.006648,13,3,0.333333,...,8.0,8.0,12.0,29.142857,28.857143,39.250000,17.295747,1,BABY CARE,2017-08-01
2,1,0,49.31,49.64,53.19,51.235,-0.006648,13,3,0.333333,...,73.0,70.0,16.0,29.857143,32.428571,41.035714,18.506112,1,BEAUTY,2017-08-01
3,1,28,49.31,49.64,53.19,51.235,-0.006648,13,3,0.333333,...,71.0,44.0,18.0,24.857143,27.785714,38.821429,19.082528,1,BEVERAGES,2017-08-01
4,1,0,49.31,49.64,53.19,51.235,-0.006648,13,3,0.333333,...,6.0,126.0,13.0,23.428571,23.571429,37.678571,19.696507,1,BOOKS,2017-08-01


8. Gắn nhãn validation gốc từ val_target_original.csv

In [18]:
target_col = None
for c in ["sales", "target", "y", "label"]:
    if c in val_target.columns:
        target_col = c
        break

if target_col is None:
    raise ValueError("Không tìm thấy cột nhãn trong val_target_original.csv (sales/target/y/label)")

val_reconstructed["val_actual_sales"] = val_target[target_col].values

print(f"Đã gắn nhãn validation từ cột: {target_col}")
display(val_reconstructed[["store_nbr", "family", "date", "val_actual_sales"]].head())

Đã gắn nhãn validation từ cột: sales


,store_nbr,family,date,val_actual_sales
0,1,AUTOMOTIVE,2017-08-01,3.0
1,1,BABY CARE,2017-08-01,0.0
2,1,BEAUTY,2017-08-01,2.0
3,1,BEVERAGES,2017-08-01,2235.0
4,1,BOOKS,2017-08-01,0.0


9. Lưu file validation đã khôi phục đầy đủ

In [19]:
val_reconstructed.to_csv(VAL_RECONSTRUCTED_PATH, index=False)

print("Đã lưu file:")
print(VAL_RECONSTRUCTED_PATH)
print("Shape:", val_reconstructed.shape)

Đã lưu file:
C:\Users\Admin\Desktop\Topic_13_Retail_Store_Sales_Time_Series\src\data\processed\splits\val_reconstructed_with_family.csv
Shape: (26730, 47)


10. Lấy các cặp (store_nbr, family) trong validation và chọn 5 cặp đại diện

In [ ]:
val_pairs = (
    val_reconstructed[["store_nbr", "family"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Số cặp (store_nbr, family) trong validation:", len(val_pairs))

sample_pairs = val_pairs.sample(n=min(5, len(val_pairs)), random_state=42).reset_index(drop=True)

print("5 cặp đại diện")
display(sample_pairs)

Số cặp (store_nbr, family) trong validation: 1782
 5 cặp đại diện


,store_nbr,family
0,31,SEAFOOD
1,5,LADIESWEAR
2,16,BEAUTY
3,31,CELEBRATION
4,30,HOME CARE


11. Lấy toàn bộ chuỗi thời gian của 5 cặp + phần validation tương ứng

In [21]:
#Toàn bộ chuỗi thời gian của 5 cặp từ train_final
representative_df = (
    train_final.merge(sample_pairs, on=["store_nbr", "family"], how="inner")
              .sort_values(["store_nbr", "family", "date"])
              .reset_index(drop=True)
)

representative_df.to_csv(REPRESENTATIVE_PATH, index=False)

print("Đã lưu chuỗi thời gian 5 cặp:")
print(REPRESENTATIVE_PATH)
print("Shape representative_5_pairs:", representative_df.shape)

#Phần validation của đúng 5 cặp đó
val_5_pairs = (
    val_reconstructed.merge(sample_pairs, on=["store_nbr", "family"], how="inner")
                    .sort_values(["store_nbr", "family", "date"])
                    .reset_index(drop=True)
)

val_5_pairs.to_csv(VAL_5_PAIRS_PATH, index=False)

print("Đã lưu validation của 5 cặp:")
print(VAL_5_PAIRS_PATH)
print("Shape val_5_pairs:", val_5_pairs.shape)

print("\nSố cặp thực tế trong representative_df:",
      representative_df[["store_nbr", "family"]].drop_duplicates().shape[0])

print("Số cặp thực tế trong val_5_pairs:",
      val_5_pairs[["store_nbr", "family"]].drop_duplicates().shape[0])

print("Số dòng validation mỗi cặp:")
display(
    val_5_pairs.groupby(["store_nbr", "family"])
              .size()
              .reset_index(name="n_val_rows")
)

Đã lưu chuỗi thời gian 5 cặp:
C:\Users\Admin\Desktop\Topic_13_Retail_Store_Sales_Time_Series\src\data\processed\splits\representative_5_pairs.csv
Shape representative_5_pairs: (8420, 51)
Đã lưu validation của 5 cặp:
C:\Users\Admin\Desktop\Topic_13_Retail_Store_Sales_Time_Series\src\data\processed\splits\val_5_pairs_reconstructed.csv
Shape val_5_pairs: (75, 47)

Số cặp thực tế trong representative_df: 5
Số cặp thực tế trong val_5_pairs: 5
Số dòng validation mỗi cặp:


,store_nbr,family,n_val_rows
0,5,LADIESWEAR,15
1,16,BEAUTY,15
2,30,HOME CARE,15
3,31,CELEBRATION,15
4,31,SEAFOOD,15
